In [4]:
import time
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from xgboost import XGBRegressor
import XG_boost as xgb_custom

housing = fetch_california_housing()
X, y = housing.data, housing.target

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Features: {housing.feature_names}")
print(f"Train shape: {X_train.shape} | Val shape: {X_val.shape}")

Features: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
Train shape: (16512, 8) | Val shape: (4128, 8)


In [6]:
params = {
    "n_estimators": xgb_custom.n_estimators,
    "learning_rate": xgb_custom.lr,
    "max_depth": xgb_custom.max_depth,
}

models = {
    "Scratch XGBoost": xgb_custom.XGBoostScratch(),
    
    "Sklearn GBDT": GradientBoostingRegressor(
        **params,
        loss="squared_error",
        random_state=42
    ),
    
    "Official XGBoost": XGBRegressor(
        **params,
        objective="reg:squarederror",
        gamma=xgb_custom.gamma,
        min_child_weight=xgb_custom.min_child_weight,
        reg_lambda=xgb_custom.Lambda,
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
        verbosity=0
    )
}

results = []
predictions = {}

for name, model in models.items():
    print(f"Running {name}...")
    
    t0 = time.time()
    model.fit(X_train, y_train)
    fit_time = time.time() - t0

    t0 = time.time()
    train_preds = model.predict(X_train)
    train_time = time.time() - t0

    t0 = time.time()
    val_preds = model.predict(X_val)
    val_time = time.time() - t0

    predictions[name] = val_preds
    val_mse = mean_squared_error(y_val, val_preds)

    results.append({
        "Model": name,
        "Fit Time (s)": fit_time,
        "Train Time (s)": train_time,
        "Val Time (s)": val_time,
        "Train MSE": mean_squared_error(y_train, train_preds),
        "Val MSE": val_mse,
        "Val RMSE": np.sqrt(val_mse),
        "Val MAE": mean_absolute_error(y_val, val_preds),
        "Train R2": r2_score(y_train, train_preds),
        "Val R2": r2_score(y_val, val_preds)
    })

results_df = pd.DataFrame(results).set_index("Model")
results_df.round(4)

Running Scratch XGBoost...
Running Sklearn GBDT...
Running Official XGBoost...


,Fit Time (s),Train Time (s),Val Time (s),Train MSE,Val MSE,Val RMSE,Val MAE,Train R2,Val R2
Model,,,,,,,,,
Scratch XGBoost,4.6903,0.1415,0.026,0.2616,0.2936,0.5418,0.3707,0.8043,0.7760
Sklearn GBDT,5.5724,0.0200,0.005,0.2608,0.2940,0.5422,0.3716,0.8049,0.7756
Official XGBoost,0.0980,0.0060,0.001,0.2613,0.2900,0.5385,0.3679,0.8045,0.7787
